In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = r"C:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\data\goldens.csv"
df.to_csv(csv_path, index=False)

In [9]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['231274cc-a223-4b81-8325-77b97d6c9d63',
  '4f5b861c-912b-462b-87c7-932702ba76be',
  'b50f5081-065f-4abc-a09d-c11a3a1963b2'],
 'count': 3,
 'as_of': '2026-05-11T11:16:09.854423107Z'}

In [3]:
from pathlib import Path
import sys
sys.path.append(str(Path("..").resolve()))
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:\\Users\\sbson\\OneDrive\\desktop\\llmops_project_agentic-based\\data\\2025_AI_Engineering_Report_Summary.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
            # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [1]:
import sys
import faiss
print("python:", sys.executable)
print("faiss module:", faiss)
print("faiss file:", getattr(faiss, "__file__", None))
print("faiss origin:", getattr(getattr(faiss, "__spec__", None), "origin", None))
print("has IndexFlatL2:", hasattr(faiss, "IndexFlatL2"))
print("dir contains IndexFlatL2:", "IndexFlatL2" in dir(faiss))

python: c:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv\Scripts\python.exe
faiss module: <module 'faiss' from 'c:\\Users\\sbson\\OneDrive\\desktop\\llmops_project_agentic-based\\.venv\\Lib\\site-packages\\faiss\\__init__.py'>
faiss file: c:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv\Lib\site-packages\faiss\__init__.py
faiss origin: c:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv\Lib\site-packages\faiss\__init__.py
has IndexFlatL2: True
dir contains IndexFlatL2: True


In [4]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-05-11T11:54:15.696786Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"keys": {}, "timestamp": "2026-05-11T11:54:15.700709Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-11T11:54:15.703710Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260511_172415_92d97954", "temp_dir": "data\\session_20260511_172415_92d97954", "faiss_dir": "faiss_index\\session_20260511_172415_92d97954", "sessionized": true, "timestamp": "2026-05-11T11:54:15.706508Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "2025_AI_Engineering_Report_Summary.txt", "saved_as": "data\\session_20260511_172415_92d97954\\117c37be.txt", "timestamp": "2026-05-11T11:54:15.711516Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-05-11T11:54:15.713636Z", "level": "info", "event": "Documents loaded"}
{"chunks": 5, "chunk_size": 10

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: OpenAI models dominate.


In [7]:
from langsmith.evaluation import evaluate, StringEvaluator

In [11]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2026-05-11T12:01:23.694998Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"keys": {}, "timestamp": "2026-05-11T12:01:23.729879Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-11T12:01:23.738862Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260511_173123_ce739db7", "temp_dir": "data\\session_20260511_173123_ce739db7", "faiss_dir": "faiss_index\\session_20260511_173123_ce739db7", "sessionized": true, "timestamp": "2026-05-11T12:01:23.753150Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "2025_AI_Engineering_Report_Summary.txt", "saved_as": "data\\session_20260511_173123_ce739db7\\7d6255ad.txt", "timestamp": "2026-05-11T12:01:23.768408Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-05-11T12:01:23.782235Z", "level": "info", "event": "Documents loaded"}
{"chunks": 5, "chunk_size": 10

Testing all questions from the dataset:



No device provided, using cpu
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: OpenAI models dominate.

--------------------------------------------------------------------------------



Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/sentence_bert_config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3118.46it/s]
HTTP Request: HEAD https://huggingface.co/sentence-

Q2: What percentage of respondents are using RAG in some form?
A2: Around 70% of the surveyed AI engineers and organizations use Retrieval-Augmented Generation (RAG) systems.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q3: How often are most respondents updating their models?
A3: More than half of the teams update their models at least monthly. Around 17% do so weekly. Continuous iteration is becoming standard in production AI systems.

--------------------------------------------------------------------------------



In [17]:
from langsmith.evaluation import evaluate, StringEvaluator
from difflib import SequenceMatcher

def simple_qa_grader(prediction: str, reference: str, _input: str | None = None) -> dict:
    pred = (prediction or "").strip().lower()
    ref = (reference or "").strip().lower()
    ratio = SequenceMatcher(None, pred, ref).ratio() if pred and ref else 0.0
    return {"score": ratio, "comment": f"similarity={ratio:.2f}"}

# Evaluators
qa_evaluator = [
    StringEvaluator(
        evaluation_name="simple_qa",
        grading_function=simple_qa_grader,
        prediction_key="answer",
        answer_key="answer",
        input_key="question",
    )
 ]
dataset_name = "AgenticAIReportGoldens"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=qa_evaluator,
    experiment_prefix="test-agenticAIReport-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-bcd1833b' at:
https://smith.langchain.com/o/a21b9683-986b-4a38-ab2d-d2394900a073/datasets/ad7c6ce8-57e5-4141-bbc4-07d149237f90/compare?selectedSessions=a85a246f-4d40-4ab5-a5d6-ecda811b0fd3




0it [00:00, ?it/s]{"timestamp": "2026-05-11T12:37:49.277986Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"keys": {}, "timestamp": "2026-05-11T12:37:49.287257Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-11T12:37:49.292729Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260511_180749_84534d22", "temp_dir": "data\\session_20260511_180749_84534d22", "faiss_dir": "faiss_index\\session_20260511_180749_84534d22", "sessionized": true, "timestamp": "2026-05-11T12:37:49.298730Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "2025_AI_Engineering_Report_Summary.txt", "saved_as": "data\\session_20260511_180749_84534d22\\46f22431.txt", "timestamp": "2026-05-11T12:37:49.301935Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-05-11T12:37:49.302758Z", "level": "info", "event": "Documents loaded"}
{"chunks": 5

Custom Correctness Evaluator
Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignment

In [18]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }

Run Evaluation with Custom Correctness Evaluator

In [19]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "AgenticAIReportGoldens"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")

View the evaluation results for experiment: 'agenticAIReport-correctness-eval-74c2e629' at:
https://smith.langchain.com/o/a21b9683-986b-4a38-ab2d-d2394900a073/datasets/ad7c6ce8-57e5-4141-bbc4-07d149237f90/compare?selectedSessions=cb170a98-6072-40e4-b613-bf9501ece205




0it [00:00, ?it/s]{"timestamp": "2026-05-11T13:36:15.480295Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"keys": {}, "timestamp": "2026-05-11T13:36:15.499798Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-11T13:36:15.502910Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260511_190615_e1d6c9ec", "temp_dir": "data\\session_20260511_190615_e1d6c9ec", "faiss_dir": "faiss_index\\session_20260511_190615_e1d6c9ec", "sessionized": true, "timestamp": "2026-05-11T13:36:15.511692Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "2025_AI_Engineering_Report_Summary.txt", "saved_as": "data\\session_20260511_190615_e1d6c9ec\\2af257e0.txt", "timestamp": "2026-05-11T13:36:15.522677Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-05-11T13:36:15.532668Z", "level": "info", "event": "Documents loaded"}
{"chunks": 5


Evaluation completed! Check the LangSmith UI for detailed results.
